# Credit Scoring Orchestrator

Notebook base para el homework. Mantiene la logica de negocio dentro de `src/`.

In [ ]:
from src.data import load_raw, validate_schema, create_features
from src.features import select_features_by_iv, build_woe_tables, transform_woe
from src.models import train_all_models, evaluate_models, save_model
from src.metrics import auc_roc, costo_total, build_scorecard

In [ ]:
df = load_raw("../data/raw/bankloan.csv")
validate_schema(df)
df = create_features(df)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(columns=["Default"]),
    df["Default"],
    test_size=0.2,
    random_state=42,
    stratify=df["Default"],
)

features_sel = select_features_by_iv(
    X_train.assign(Default=y_train),
    target="Default",
    threshold=0.1,
)
features_sel

In [ ]:
woe_tables = build_woe_tables(
    X_train.assign(Default=y_train),
    features_sel,
    target="Default",
)

X_train_woe = transform_woe(X_train[features_sel], woe_tables)
X_test_woe = transform_woe(X_test[features_sel], woe_tables)
X_train_woe.head()

In [ ]:
models = train_all_models(X_train_woe, y_train)
tabla_auc = evaluate_models(models, X_test_woe, y_test)
tabla_auc

In [ ]:
mejor_nombre = tabla_auc.iloc[0]["Modelo"]
mejor_modelo = models[mejor_nombre]

metadata = {
    "model_name": mejor_nombre,
    "version": "1.0",
    "author": "Arenas",
    "dataset": "data/raw/bankloan.csv",
    "n_train_samples": int(len(y_train)),
    "n_test_samples": int(len(y_test)),
    "features_selected": features_sel,
    "hyperparameters": mejor_modelo.get_params(),
    "metrics": {
        "auc_test": round(auc_roc(mejor_modelo, X_test_woe, y_test), 4),
        "costo_total_test": int(
            costo_total(
                y_test.values,
                mejor_modelo.predict_proba(X_test_woe)[:, 1],
                umbral=0.5,
            )
        ),
    },
}

save_model(mejor_modelo, path="../models/baseline_v1", metadata=metadata)

In [ ]:
import json
from pathlib import Path

metadata_path = Path("../models/baseline_v1/metadata.json")

assert metadata_path.exists(), "metadata.json no existe - revisa save_model()"

with open(metadata_path, encoding="utf-8") as f:
    meta = json.load(f)

campos_requeridos = [
    "model_name",
    "version",
    "saved_at",
    "author",
    "dataset",
    "n_train_samples",
    "n_test_samples",
    "features_selected",
    "hyperparameters",
    "metrics",
]

for campo in campos_requeridos:
    assert campo in meta, f"Campo faltante en metadata.json: {campo}"

meta